# Stage 4 — Baseline + Merge + Experiments


## 1. Load data

In [ ]:

import pandas as pd
from pathlib import Path

ROOT = Path("../")

emotion_path = ROOT / "datasets/metadata/pilot_video_emotion_features.csv"
manifest_path = ROOT / "datasets/metadata/pilot_manifest.csv"
detector_path = ROOT / "datatsets/metadata/pilot_detector_scores.csv"

emotion_df = pd.read_csv(emotion_path)
manifest_df = pd.read_csv(manifest_path)
detector_df = pd.read_csv(detector_path)

print(len(emotion_df), len(manifest_df), len(detector_df))


FileNotFoundError: [Errno 2] No such file or directory: '/home/n.salikhova@innopolis.university/deepfake-emotion-robustness/datasets/metadata/final_video_emotion_features.csv'

## 2. Merge

In [ ]:

df = manifest_df.merge(emotion_df, on="video_id")
df = df.merge(detector_df, on="video_id")

print(df.shape)
df.head()


## 3. Metrics (baseline)

In [ ]:

from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

y_true = df["label"]
y_pred = df["detector_score"]

auc = roc_auc_score(y_true, y_pred)
acc = accuracy_score(y_true, (y_pred > 0.5).astype(int))
f1 = f1_score(y_true, (y_pred > 0.5).astype(int))

print("AUC:", auc)
print("ACC:", acc)
print("F1:", f1)


## 4. Emotion analysis

In [ ]:

df["arousal_bin"] = pd.qcut(df["mean_arousal"], 3, labels=["low", "mid", "high"])

for g in df["arousal_bin"].unique():
    sub = df[df["arousal_bin"] == g]
    auc = roc_auc_score(sub["label"], sub["detector_score"])
    print(g, "AUC:", auc, "n=", len(sub))


## 5. Fusion model

In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

features = [
    "detector_score",
    "mean_arousal",
    "std_arousal",
    "mean_valence",
    "emotion_entropy",
    "transition_rate"
]

X = df[features].fillna(0)
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict_proba(X_test)[:, 1]

from sklearn.metrics import roc_auc_score
print("Fusion AUC:", roc_auc_score(y_test, y_pred))


## 6. Compare

In [ ]:

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

baseline_auc = roc_auc_score(df["label"], df["detector_score"])

emotion_features = df[[
    "mean_arousal",
    "std_arousal",
    "mean_valence",
    "emotion_entropy",
    "transition_rate"
]].fillna(0)

model2 = LogisticRegression(max_iter=1000)
model2.fit(emotion_features, df["label"])
emotion_pred = model2.predict_proba(emotion_features)[:, 1]

emotion_auc = roc_auc_score(df["label"], emotion_pred)

print("Baseline AUC:", baseline_auc)
print("Emotion AUC:", emotion_auc)
